# Lab 011 — Read a tree split on real credit data

**Lesson:** [`lessons/0011-decision-trees-partitions.html`](../lessons/0011-decision-trees-partitions.html) · **Phase / Year:** Year 1 · Q2

**Dataset tier:** A — OpenML German credit (`credit_g`) via `relkit`. Real mixed-type tabular data; default rate ~70% so you see impurity on an imbalanced task.

**Skill you are practising:** fit a depth-1 decision tree on real data, read its partition rule, and verify the Gini impurity reduction for that split.

**Exit criteria:** EXIT TICKET prints split feature name, threshold, child class rates, and hand-computed ΔGini matching sklearn within tolerance.

---

### How this notebook works

- **PROVIDED** cells — complete boilerplate; just run.
- **TODO** cells — blanks (`____` / `# TODO`); you implement the skill.
- **CHECK** cells — immediate feedback; do not edit.
- Run top to bottom. When **EXIT TICKET** prints cleanly, paste it to your teacher or say *"lab done"*.

### Environment

One-time: `bash labs/setup-env.sh` from repo root → kernel **Relational Labs (.venv)**.


## Concept recap — decision trees as partitions

A **classification tree** recursively splits the feature space. At each node it picks **one feature** `j` and **one threshold** `t`, then sends rows left if `x[j] ≤ t`, else right. Every split is **axis-aligned** (parallel to a feature axis) — in 2D, a vertical or horizontal line, never diagonal.

A **depth-1 tree** asks a single question and creates **two regions**. Each leaf predicts the **majority class** of the rows that land there. That is a **piecewise-constant** model — the same picture your Q1 `HistGradientBoostingClassifier` uses, but with hundreds of small regions instead of one.

**What the algorithm optimizes:** at each candidate split it maximizes **impurity reduction** (Gini drop). For binary labels in `{0, 1}` with `p = P(y=1)`:

$$G = 1 - p^2 - (1-p)^2 = 2p(1-p)$$

- Pure node (all 0 or all 1): `G = 0`
- 50/50 mix: `G = 0.5` (worst for binary)

After splitting into left/right with row fractions `w_L`, `w_R`:

$$\Delta G = G_{\text{parent}} - \big(w_L \cdot G_{\text{left}} + w_R \cdot G_{\text{right}}\big)$$

**Toy example (not this dataset):** labels `[0, 0, 1, 1]` → `p = 0.5` → `G = 0.5`. Split left=`[0,0]`, right=`[1,1]` → children pure → `ΔG = 0.5`. A useless split that leaves both children as mixed as the parent gives `ΔG ≈ 0`.

**In this lab:** sklearn picks the best depth-1 split on German credit. You read `tree_.feature` / `tree_.threshold`, inspect child class rates, then **hand-compute ΔG** to prove you understand the partition rule — the same logic XGBoost scales up with regularized gain.

Full viz and reading: [Lesson 011](../lessons/0011-decision-trees-partitions.html) · ESL Ch. 9.2.

## Setup — PROVIDED


In [4]:
# PROVIDED
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))  # labs/))

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

from relkit.data import load_tier_a

RS = 0
print("relkit ok")


relkit ok


## Data — PROVIDED (Tier A)

**What you get:** 1000 rows of real German credit applications — numeric columns (loan amount, duration, …) plus categoricals (checking status, purpose, …). Labels: good (`0`) vs bad (`1`) credit risk.

**Why encode categoricals?** `DecisionTreeClassifier` needs numeric inputs. We label-encode strings to integers **for this partition demo only** — CatBoost (Lesson 016) handles categories natively; for production baselines you'd use a proper `ColumnTransformer` inside a pipeline (Lesson 005).

Run the cell below to load via `relkit` and print row count, positive rate, and feature count.


In [5]:
# PROVIDED
X, y = load_tier_a("credit_g")
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_num = X[num_cols].copy()
for c in cat_cols:
    X_num[c] = LabelEncoder().fit_transform(X[c].astype(str))

feat_names = list(X_num.columns)
print(f"rows={len(y)}  pos_rate={y.mean():.3f}  features={len(feat_names)}")


rows=1000  pos_rate=0.700  features=20


## Task 1 — Fit depth-1 tree — TODO

**Goal:** one fitted tree and its split rule printed as `feature <= threshold`.

**Why it matters:** before tuning ensembles (XGBoost, LightGBM), you must read what a *single* split does — which feature and cutoff carve the space.

**You implement:**
1. `DecisionTreeClassifier(max_depth=1, random_state=RS)` on `X_num`, `y`
2. Read root split from `tree.tree_`: `feature[0]`, `threshold[0]`, map index → name via `feat_names`


In [6]:
# TODO
tree = DecisionTreeClassifier(max_depth=1, random_state=RS)
tree.fit(X_num, y)

t = tree.tree_
split_feat_idx = t.feature[0]
split_thresh = t.threshold[0]
split_name = feat_names[split_feat_idx]
print(f"split: {split_name} <= {split_thresh:.4f}")


split: checking_status <= 1.5000


In [7]:
# CHECK
assert tree.get_depth() == 1, "Use max_depth=1."
assert split_feat_idx >= 0, "Tree must pick a split feature."
print("Task 1 ok — depth-1 tree fitted.")


Task 1 ok — depth-1 tree fitted.


## Task 2 — Inspect the partition — TODO

**Goal:** row counts and positive rates in the left vs right child of the root split.

**Why it matters:** a tree is a partition — you need to see *who landed where* before judging whether the split purified the classes.

**You implement:** boolean masks `left_mask` / `right_mask` from the split feature and threshold; compute `y[left_mask].mean()` and `y[right_mask].mean()` (positive rate = `P(y=1)` in each region).

**sklearn node layout:** node 0 = root; node 1 = left child; node 2 = right child in `tree_.impurity`.


In [8]:
# TODO — majority class in left/right child (0 or 1)
left_mask = X_num.iloc[:, split_feat_idx] <= split_thresh
right_mask = ~left_mask

left_rate = y[left_mask].mean()
right_rate = y[right_mask].mean()
print(f"left  n={left_mask.sum():4d}  pos_rate={left_rate:.3f}")
print(f"right n={right_mask.sum():4d}  pos_rate={right_rate:.3f}")


left  n= 543  pos_rate=0.558
right n= 457  pos_rate=0.869


In [9]:
# CHECK
assert left_mask.sum() + right_mask.sum() == len(y)
print("Task 2 ok — partition covers all rows.")


Task 2 ok — partition covers all rows.


## Task 3 — Hand-compute ΔGini — TODO (crucial fragment)

**Goal:** implement `gini(y_vec)` and compute `delta_g` for the split above; CHECK compares to sklearn's `tree_.impurity`.

**Why it matters:** libraries hide the objective. Hand-computing ΔG proves you know what "best split" *means* — weighted child purity gain — not just `fit()`.

**You implement:**
1. `gini`: for binary `y`, return `1 - p**2 - (1-p)**2` where `p = y_vec.mean()` (return `0.0` on empty)
2. `G_parent`, `G_left`, `G_right` on full / left / right labels
3. `w_left`, `w_right` = fraction of rows in each child
4. `delta_g = G_parent - (w_left * G_left + w_right * G_right)` ← parent **minus** weighted children

**Common mistake:** using `p` (positive rate) instead of Gini, or forgetting to subtract from `G_parent`.


In [14]:
def gini(y_vec):
    """Gini impurity for binary labels in {0,1}."""
    if len(y_vec) == 0:
        return 0.0
    p = y_vec.mean()
    return ____  # TODO: 1 - p**2 - (1-p)**2

# TODO: parent and weighted child Gini, then impurity reduction
G_parent = ____
G_left = ____
G_right = ____
w_left = ____
w_right = ____
delta_g = ____  # TODO: G_parent - (w_left * G_left + w_right * G_right)

print(f"G_parent={G_parent:.4f}  delta_gini={delta_g:.4f}")


G_parent=0.4200  delta_gini=0.2564


In [15]:
# CHECK
assert 0 <= G_parent <= 0.5, "Binary Gini should be in [0, 0.5]."
assert delta_g > 0, "The chosen split should reduce impurity."
sklearn_imp = tree.tree_.impurity[0] - (
    (left_mask.sum()/len(y))*tree.tree_.impurity[1]
    + (right_mask.sum()/len(y))*tree.tree_.impurity[2]
)
assert abs(delta_g - sklearn_imp) < 0.02, f"Hand ΔGini {delta_g:.4f} vs sklearn {sklearn_imp:.4f}"
print("Task 3 ok — hand ΔGini matches sklearn.")


AssertionError: Hand ΔGini 0.2564 vs sklearn 0.0479

## Exit ticket — TODO

**Goal:** one printed summary you paste back to your teacher.

**Takeaway prompt:** in one sentence, connect the split you found to the partition picture — e.g. which feature/threshold separated higher-risk rows, and whether ΔGini confirms sklearn picked a purifying split.


In [ ]:
# TODO: complete takeaway
print("=== EXIT TICKET — Lesson 011 ===")
print(f"split feature : {split_name}")
print(f"threshold     : {split_thresh:.4f}")
print(f"left pos_rate : {left_rate:.3f}  right pos_rate: {right_rate:.3f}")
print(f"delta Gini    : {delta_g:.4f}")
print()
print("takeaway:", "____")
